In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.impute import SimpleImputer

# Load the merged data
print("Loading data...")
data = pd.read_csv('~/Desktop/merged_bushfire_weather_data.csv')

# Feature engineering
def prepare_features(df):
    print("Preparing features...")
    # Convert date to datetime and extract useful features
    df['discovery_date'] = pd.to_datetime(df['discovery_date'])
    df['month'] = df['discovery_date'].dt.month
    df['day_of_year'] = df['discovery_date'].dt.dayofyear
    
    # Calculate temperature range
    df['temp_range'] = df['t_max'] - df['t_min']
    
    # You can add more feature engineering steps here
    return df

data = prepare_features(data)

# Define features and target
features = ['latitude', 'longitude', 't_min', 't_max', 'elevation', 'temp_range', 'month', 'day_of_year']
target = 'fire_size'  # Assuming fire_size as a proxy for risk

# Prepare the feature matrix X and target vector y
X = data[features]
y = data[target]

# Split the data
print("Splitting data into train and test sets...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Impute missing values
print("Imputing missing values...")
imputer = SimpleImputer(strategy='mean')
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# Scale features
print("Scaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the model
print("Training the model...")
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# Make predictions
print("Making predictions...")
y_pred = model.predict(X_test_scaled)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared Score: {r2}")

# Feature importance
feature_importance = pd.DataFrame({'feature': features, 'importance': model.feature_importances_})
print("\nFeature Importance:")
print(feature_importance.sort_values('importance', ascending=False))

# Function to predict risk for a given location and conditions
def predict_risk(lat, lon, t_min, t_max, elevation, date):
    # Prepare input data
    input_data = pd.DataFrame({
        'latitude': [lat],
        'longitude': [lon],
        't_min': [t_min],
        't_max': [t_max],
        'elevation': [elevation],
        'temp_range': [t_max - t_min],
        'month': [date.month],
        'day_of_year': [date.timetuple().tm_yday]
    })
    
    # Scale input data
    input_scaled = scaler.transform(input_data)
    
    # Make prediction
    risk = model.predict(input_scaled)[0]
    
    # Normalize risk to 0-1 range
    max_fire_size = y.max()
    normalized_risk = min(risk / max_fire_size, 1)
    
    return normalized_risk


Loading data...
Preparing features...
Splitting data into train and test sets...
Imputing missing values...
Scaling features...
Training the model...


KeyboardInterrupt: 

In [ ]:

# Example usage
from datetime import datetime
risk = predict_risk(lat=34.0522, lon=-118.2437, t_min=15, t_max=30, elevation=100, date=datetime(2025, 5, 15))
print(f"\nPredicted bushfire risk: {risk:.4f}")